In [1]:
import os
import re
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# --- 1. CHARGEMENT ET PRÉPARATION DES DONNÉES ---
data_path = '../data/processed/cleaned_data.csv'

print("--- Chargement du fichier cleaned_data.csv ---")
data = pd.read_csv(data_path)

print(f"Forme du jeu de données (Data shape) : {data.shape}")

# Dans votre dataset, la colonne de texte est 'email_text'
print(f"Valeurs manquantes dans 'email_text' : {data['email_text'].isnull().sum()}")

# Traitement des valeurs NaN au cas où
data['email_text'] = data['email_text'].fillna('')

# --- 2. FONCTION DE PRÉTRAITEMENT (Inspirée du projet GitHub) ---
def preprocess(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'<.*?>', '', text)  # Supprimer les balises HTML
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Supprimer les liens
        text = re.sub(r'[^a-zA-Z\s]', '', text)  # Supprimer les caractères spéciaux et chiffres
        text = re.sub(r'\s+', ' ', text).strip()  # Nettoyer les espaces multiples
        return text
    else:
        return ''

print("\nStarting text preprocessing...")
# Vos données ont déjà été nettoyées dans le notebook 01, 
# mais nous appliquons cette étape pour assurer la compatibilité exacte avec GitHub.
data['cleaned_body'] = data['email_text'].apply(preprocess)
print("Completed text preprocessing.")

# --- 3. VECTORISATION TF-IDF ---
print("\n--- Vectorisation TF-IDF ---")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

X = tfidf_vectorizer.fit_transform(data['cleaned_body'])
y = data['label']

print("TF-IDF vectorization complete.")

# --- 4. SÉPARATION TRAIN / TEST ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Data split into training and testing sets.")

# --- 5. ENTRAÎNEMENT DU MODÈLE (RandomForestClassifier) ---
print("\nStarting model training (Random Forest)...")
# n_estimators=100 offre de bien meilleures performances que 10
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

model.fit(X_train, y_train)
print("Model training complete.")

# --- 6. ÉVALUATION DU MODÈLE ---
y_pred = model.predict(X_test)

print("\n--- Matrice de confusion ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Rapport de classification ---")
print(classification_report(y_test, y_pred))

# --- 7. SAUVEGARDE DU MODÈLE ET DU VECTORISEUR ---
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, 'phishing_detection_model.pkl')
vectorizer_path = os.path.join(models_dir, 'tfidf_vectorizer.pkl')

joblib.dump(model, model_path)
joblib.dump(tfidf_vectorizer, vectorizer_path)

print(f"\nModel saved to: {model_path}")
print(f"Vectorizer saved to: {vectorizer_path}")

--- Chargement du fichier cleaned_data.csv ---
Forme du jeu de données (Data shape) : (82484, 3)
Valeurs manquantes dans 'email_text' : 0

Starting text preprocessing...
Completed text preprocessing.

--- Vectorisation TF-IDF ---
TF-IDF vectorization complete.
Data split into training and testing sets.

Starting model training (Random Forest)...
Model training complete.

--- Matrice de confusion ---
[[7809  110]
 [ 191 8387]]

--- Rapport de classification ---
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      7919
           1       0.99      0.98      0.98      8578

    accuracy                           0.98     16497
   macro avg       0.98      0.98      0.98     16497
weighted avg       0.98      0.98      0.98     16497


Model saved to: ../models\phishing_detection_model.pkl
Vectorizer saved to: ../models\tfidf_vectorizer.pkl
